# Predict the electricity price

* Features:
    - electricy demand prediction
    - PV generation prediction
    - wind generation prediction
    - residual load (PV generation + wind generation - electricity demand)
*

In [1]:
# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

import sys
import os

# Add the src directory to the system path to allow importing custom modules
project_root = os.path.abspath("..")
src_dir = os.path.join(project_root, "src")
util_dir = os.path.join(project_root, "util")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
if util_dir not in sys.path:
    sys.path.insert(0, util_dir)


import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2
    
from src.config import *
from util.weather_weighted import *
from src.etl_price import *
from src.fetch_price_data import *
from src.train_predict_model import *
from src.config import DATABASE_PATH, DEMAND_UPSTREAM_MODEL_PATH
from src.etl_price import create_price_tables, seed_series_catalog
from src.historical_price_weather import (
    fetch_and_store_demand_weather_for_target, fetch_and_store_weather_for_target,
)
from src.price_walk_forward import predict_price_target_day_from_db
from src.price_prediction_store import create_prediction_run, store_versioned_price_predictions
from util.time_features import *

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import hashlib

# One-liner data preparation: loads, merges, engineers features, returns training-ready dataset 
df_price_model = prepare_price_model_dataset()

print("model rows:", len(df_price_model))
print("time range:", df_price_model["time"].min(), "->", df_price_model["time"].max())
print("feature count:", len(df_price_model.columns) - 2)  # Subtracting 'time' and target column
display(df_price_model.shape)

# The static feature table is used for training only.  Its observed inputs
# must not be used to score future delivery days.  Evaluation below goes
# through the leakage-safe walk-forward pipeline.
training_cutoff = pd.Timestamp("2025-10-01", tz="Europe/Berlin")
df_price_train = df_price_model.loc[df_price_model["time"] < training_cutoff].copy()
price_features_train = df_price_train.drop(columns=["time", "price_de_lu_eur_mwh"])
price_target_train = df_price_train["price_de_lu_eur_mwh"]

print("Training cutoff (exclusive):", training_cutoff)
print("Training set:", price_features_train.shape, price_target_train.shape)

In [4]:
# tune LightGBM model with Bayesian optimization
from lightgbm import LGBMRegressor
from skopt import BayesSearchCV

param_lgbm = {
    'n_estimators':  (50, 500),
    'learning_rate': (0.01, 0.3),
    'max_depth':     (3, 15),
}

start_time = pd.Timestamp.now()

model_lgbm = LGBMRegressor(random_state=42, force_col_wise=True)
price_model_lgbm, best_params_lgbm = tune_model_bayesian(
    model_pipeline=model_lgbm,
    in_param_bayes=param_lgbm,
    in_features_train=price_features_train,
    in_target_train=price_target_train,
    mlflow_experiment="electricity-price",
    mlflow_run_name="lgbm-bayesian-tuning",
    mlflow_tags={"dataset": "price_train_test", "model_family": "lightgbm"},
)
print(f"Best hyperparameters: {best_params_lgbm}")
print()

save_model_to_pickle(price_model_lgbm, '../models/price_lgbm_model.pkl')
print("Saved to: ../models/price_lgbm_model.pkl")

# training time
end_time = pd.Timestamp.now()
training_time = end_time - start_time
# show in minutes and seconds
minutes, seconds = divmod(training_time.total_seconds(), 60)
print(f"\nTraining time: {int(minutes)}'{int(seconds)}\"")

In [5]:
# Tune the XGBoost model using the standardized feature set
from xgboost import XGBRegressor

param_xgb_continuous = {
    'n_estimators': (50, 1000),
    'max_depth': (3, 15),
    'learning_rate': (0.01, 0.3),
    'subsample': (0.5, 1.0),
    'colsample_bytree': (0.5, 1.0),
}

start_time = pd.Timestamp.now()

model_xgb = XGBRegressor(random_state=42)
price_model_xgb, price_best_params_xgb = tune_model_bayesian(
    model_pipeline=model_xgb,
    in_param_bayes=param_xgb_continuous,
    in_features_train=price_features_train,   
    in_target_train=price_target_train,
    mlflow_experiment="electricity-price",
    mlflow_run_name="xgb-bayesian-tuning",
    mlflow_tags={"dataset": "price_train_test", "model_family": "xgboost"}
)
print(f"Best hyperparameters for XGBoost: {price_best_params_xgb}")

save_model_to_pickle(price_model_xgb, '../models/price_xgb_model.pkl')

# training time
end_time = pd.Timestamp.now()
training_time = end_time - start_time
# show in minutes and seconds
minutes, seconds = divmod(training_time.total_seconds(), 60)
print(f"\nTraining time: {int(minutes)}'{int(seconds)}\"")

In [6]:
# Leakage-safe evaluation.  Every delivery day D is predicted solely from
# inputs available at D-1 11:30 Europe/Berlin; actual physical values are
# masked after D-2 by src.forecast_protocol.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

EVALUATION_START = "2025-10-01"
EVALUATION_END = "2025-10-07"  # extend deliberately; weather runs are fetched per day
MODEL_TO_EVALUATE = ("XGBoost", price_model_xgb)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

frozen_demand_model = load_model_from_pickle(DEMAND_UPSTREAM_MODEL_PATH)
conn = create_price_tables(DATABASE_PATH)
seed_series_catalog(conn)
run_id = create_prediction_run(
    conn, model_name=MODEL_TO_EVALUATE[0],
    model_sha256=sha256_file("../models/price_xgb_model.pkl"),
    demand_model_sha256=sha256_file(DEMAND_UPSTREAM_MODEL_PATH),
    feature_pipeline_version="calendar-lags-v1",
    protocol_version="price-walk-forward-v1",
    evaluation_mode="walk_forward_as_of_d1_1130",
)
print(f"Persisting versioned predictions under run_id={run_id}")
predictions = []
try:
    for target_day in pd.date_range(EVALUATION_START, EVALUATION_END, freq="D"):
        target_date = target_day.strftime("%Y-%m-%d")
        # Persist the historic ECMWF runs that were available at the as-of time.
        fetch_and_store_weather_for_target(conn, target_date)
        fetch_and_store_demand_weather_for_target(conn, target_date)
        prediction = predict_price_target_day_from_db(
            MODEL_TO_EVALUATE[1], frozen_demand_model, conn, target_date, MODEL_TO_EVALUATE[0]
        )
        store_versioned_price_predictions(conn, run_id, prediction)
        predictions.append(prediction)
finally:
    conn.close()

walk_forward = pd.concat(predictions, ignore_index=True).dropna(subset=["actual_eur_mwh"])
actual = walk_forward["actual_eur_mwh"]
prediction = walk_forward["prediction_eur_mwh"]
scores = pd.Series({
    "MAE": mean_absolute_error(actual, prediction),
    "RMSE": mean_squared_error(actual, prediction) ** 0.5,
    "R²": r2_score(actual, prediction),
    "hours": len(walk_forward),
})
display(scores.to_frame(name=MODEL_TO_EVALUATE[0]))
display(walk_forward[["target_time", "as_of_time", "actual_eur_mwh", "prediction_eur_mwh"]])

In [6]:
# learn curve for LightGBM and XGBoost model
plot_learning_curve(price_model_lgbm, 'LightGBM', price_features_train, price_target_train)
plot_learning_curve(price_model_xgb,  'XGBoost', price_features_train, price_target_train)
